In [1]:
import sys
sys.path.append("..")  # let the notebook import modules from the project root

import pandas as pd
from src.load_data import load_recalls

df = load_recalls()
veh = df[df["RCLTYPECD"] == "V"].copy()  # vehicles only

print("All rows:", len(df))
print("Vehicle rows:", len(veh))
print("Vehicle recalls:", veh["CAMPNO"].nunique())

All rows: 246899
Vehicle rows: 220165
Vehicle recalls: 13721


In [2]:
# Is COMPNAME / DESC_DEFECT the same on every row of a recall?
for col in ["COMPNAME", "DESC_DEFECT", "CONEQUENCE_DEFECT"]:
    share = (veh.groupby("CAMPNO")[col].nunique() == 1).mean().round(3)
    print(f"{col}: {share}")

COMPNAME: 0.843
DESC_DEFECT: 1.0
CONEQUENCE_DEFECT: 1.0


In [3]:
# Which component names mention batteries? (count = number of recalls)
batt = veh.loc[veh["COMPNAME"].str.contains("BATTER", na=False), ["CAMPNO", "COMPNAME"]]
batt.drop_duplicates()["COMPNAME"].value_counts().head(30)

COMPNAME
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY                                                                121
ELECTRICAL SYSTEM:12V/24V/48V BATTERY:CABLES                                                                         98
ELECTRICAL SYSTEM:12V/24V/48V BATTERY                                                                                83
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY:MANAGEMENT SYSTEM/ENERGY CONTROL MODULE (BMS/BECM):SOFTWARE     15
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY:THERMAL MANAGEMENT:FLUID/COOLANT                                10
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY:MANAGEMENT SYSTEM/ENERGY CONTROL MODULE (BMS/BECM)               7
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY THERMAL:MANAGEMENT:SOFTWARE                                      6
EQUIPMENT:RECREATIONAL VEHICLE/TRAILER:120/240 VAC ELECTRICAL SYSTEM:CHARGER LOW VOLTAGE BATTERY                      1
ELECTRICAL SYSTEM:PROPULSION SY

In [4]:
# Row-level component flags
veh["hv_comp"] = veh["COMPNAME"].str.contains("TRACTION BATTERY", na=False)
veh["lv_comp"] = veh["COMPNAME"].str.contains("12V/24V/48V BATTERY", na=False)

# Collapse to one row per recall: a flag is True if ANY row of the recall has it
rec = veh.groupby("CAMPNO").agg(
    hv_comp=("hv_comp", "any"),
    lv_comp=("lv_comp", "any"),
    MFGNAME=("MFGNAME", "first"),
    RCDATE=("RCDATE", "first"),
    POTAFF=("POTAFF", "first"),
    DESC_DEFECT=("DESC_DEFECT", "first"),
    CONEQUENCE_DEFECT=("CONEQUENCE_DEFECT", "first"),
)
print("Recalls:", len(rec))

# Sanity check: are MFGNAME and RCDATE constant within a recall (so "first" is safe)?
for col in ["MFGNAME", "RCDATE"]:
    print(col, (veh.groupby("CAMPNO")[col].nunique() == 1).mean().round(3))

Recalls: 13721
MFGNAME 1.0
RCDATE 1.0


In [5]:
BATTERY_TEXT = r"\bBATTER(?:Y|IES)\b|\bLITHIUM\b|HIGH[- ]VOLTAGE|THERMAL RUNAWAY|ELECTROLYTE"
FIRE_TEXT = r"\bFIRES?\b|THERMAL RUNAWAY"

text = rec["DESC_DEFECT"] + " " + rec["CONEQUENCE_DEFECT"]
rec["battery_text"] = text.str.contains(BATTERY_TEXT, case=False, regex=True)
rec["fire_text"] = text.str.contains(FIRE_TEXT, case=False, regex=True)

rec["battery_any"] = rec["hv_comp"] | rec["lv_comp"] | rec["battery_text"]
rec["battery_fire"] = rec["battery_any"] & rec["fire_text"]

rec[["hv_comp", "lv_comp", "battery_text", "fire_text", "battery_any", "battery_fire"]].sum()

hv_comp          158
lv_comp          181
battery_text     558
fire_text       2967
battery_any      582
battery_fire     347
dtype: int64

In [6]:
pd.set_option("display.max_colwidth", 300)
rec.loc[rec["battery_fire"], ["MFGNAME", "hv_comp", "CONEQUENCE_DEFECT"]].sample(10, random_state=42)

,MFGNAME,hv_comp,CONEQUENCE_DEFECT
CAMPNO,,,
26V001000,"Volvo Car USA, LLC",True,A high-voltage battery that overheats increases the risk of a fire.
18V553000,Altec Industries Inc.,False,An electrical short can increase the risk of a fire.
26V062000,Ford Motor Company,True,"Electrical arcing increases the risk of a fire. In addition, a loss of drive power may occur, which can increase the risk of a crash."
24V726000,"Volkswagen Group of America, Inc.",True,A short circuit in a high-voltage battery module increases the risk of a fire.
25V567000,Motor Coach Industries,True,A battery module that overheats increases the risk of a fire.
19V485000,Oshkosh Corporation,False,"If the battery cable continues to rub against the fuse box bracket, it may short-circuit, increasing the risk of fire."
23V329000,"Ferrari North America, Inc.",False,A fuel leak in the presence of an ignition source increases the risk of a fire.
23V734000,Toyota Motor Engineering & Manufacturing,False,A battery short circuit increases the risk of a fire.
24V395000,"Forest River, Inc.",False,"A loose wire can cause a short circuit, increasing the risk of a fire."


In [7]:
# Read the full defect description for the unclear cases
for campno in ["23V329000", "18V553000", "24V395000"]:
    print(campno, "->", rec.loc[campno, "DESC_DEFECT"], "\n")

23V329000 -> Ferrari North America, Inc. (Ferrari) is recalling certain 2022-2023 Ferrari 296 vehicles.  The fuel tank connecting pipe may corrode due to unintended contact between the fuel tank connecting pipe and the high-voltage battery protection cover, potentially resulting in a fuel leak. 

18V553000 -> Altec Industries Inc. (Altec) is recalling certain 2017-2018 Altec LS63 Aerial Devices.  The fuse for the battery was not installed to the guidelines.  The incorrectly placed fuse may increase the risk for an electrical short. 

24V395000 -> Forest River, Inc. (Forest River) is recalling certain 2004-2024 Cedar Creek and 2006-2024 Puma fifth wheels trailers.  The overcurrent protection from the trailer battery may have been incorrectly routed, allowing the wires to be pulled out of the junction box.  



In [8]:
# How many traction-battery recalls mention fire, and how many don't?
pd.crosstab(rec["hv_comp"], rec["fire_text"], margins=True)

fire_text,False,True,All
hv_comp,,,
False,10707,2856,13563
True,47,111,158
All,10754,2967,13721


In [9]:
# What are the 47 traction-battery recalls that do NOT mention fire?
rec.loc[rec["hv_comp"] & ~rec["fire_text"], "CONEQUENCE_DEFECT"].sample(8, random_state=42)

CAMPNO
23V295000                                                                       An engine stall increases the risk of a crash.
24V816000                                                      An engine that fails to restart increases the risk of a crash. 
23V095000                                                         A sudden loss of drive power increases the risk of a crash. 
25V441000    An overheated high voltage battery contactor can result in a loss of drive power, increasing the risk of a crash.
22V944000      An interruption of electrical power may result in a sudden loss of drive power, increasing the risk of a crash.
24V475000                                                                A loss of drive power increases the risk of a crash. 
21V869000                                                          A sudden loss of drive power increases the risk of a crash.
22V876000                 An unexpected shut down can result in a sudden loss of drive power, increasing

In [10]:
import numpy as np

# One battery category per recall (HV takes priority if both are present)
rec["battery_type"] = np.select(
    [rec["hv_comp"], rec["lv_comp"]],
    ["HV traction battery", "12V/24V/48V battery"],
    default="Battery (text only)",
)
bf = rec[rec["battery_fire"]].copy()   # our main analysis set
bf["battery_type"].value_counts()

battery_type
12V/24V/48V battery    132
HV traction battery    111
Battery (text only)    104
Name: count, dtype: int64

In [11]:
# Convert text columns to proper types
bf["RCDATE"] = pd.to_datetime(bf["RCDATE"], format="%Y%m%d", errors="coerce")
bf["POTAFF"] = pd.to_numeric(bf["POTAFF"], errors="coerce")
bf["year"] = bf["RCDATE"].dt.year

print("Bad dates:", bf["RCDATE"].isna().sum())
print("Bad POTAFF:", bf["POTAFF"].isna().sum())
print(bf["POTAFF"].describe())

Bad dates: 0
Bad POTAFF: 0
count    3.470000e+02
mean     1.928295e+04
std      1.265949e+05
min      1.000000e+00
25%      4.100000e+01
50%      2.640000e+02
75%      2.794500e+03
max      1.853568e+06
Name: POTAFF, dtype: float64


In [12]:
# Which recalls dominate the affected-vehicle total?
cols = ["MFGNAME", "year", "battery_type", "POTAFF", "CONEQUENCE_DEFECT"]
bf.nlargest(5, "POTAFF")[cols]

,MFGNAME,year,battery_type,POTAFF,CONEQUENCE_DEFECT
CAMPNO,,,,,
23V734000,Toyota Motor Engineering & Manufacturing,2023,12V/24V/48V battery,1853568,A battery short circuit increases the risk of a fire.
17V418000,Honda (American Honda Motor Co.),2017,12V/24V/48V battery,1148550,An electrical short increases the risk of a fire.
22V024000,"Nissan North America, Inc.",2022,Battery (text only),664779,"Corrosion in the electrical connector can result in power window/seat failure, all-wheel-drive warning light illumination, battery drainage, and electrical connector damage that increases the risk of a fire."
12V126000,"BMW OF NORTH AMERICA, LLC",2012,12V/24V/48V battery,367682,"OVERHEATING COULD LEAD TO A FIRE, EVEN WHILE THE VEHICLE IS NOT IN USE. AS A PRECAUTION, OWNERS ARE ADVISED TO PARK OUTSIDE UNTIL THE REMEDY HAS BEEN MADE."
19V888000,General Motors LLC,2019,12V/24V/48V battery,350371,"An intermittent electrical connection between the B+ battery cable and the alternator could cause the vehicle to stall, increasing the risk of a crash. The intermittent connection could also lead to electrical arcing, which could generate sufficient heat to damage surrounding material, increasi..."


In [13]:
# How many distinct manufacturer names, and do variants of the same company exist?
print("Distinct MFGNAME:", bf["MFGNAME"].nunique())
bf["MFGNAME"].value_counts().head(30)

Distinct MFGNAME: 141


MFGNAME
Ford Motor Company                      20
BMW of North America, LLC               17
Volkswagen Group of America, Inc.       15
Forest River, Inc.                      13
Jaguar Land Rover North America, LLC    10
Chrysler (FCA US, LLC)                   8
Volvo Trucks North America               7
Motor Coach Industries                   7
Daimler Trucks North America LLC         7
General Motors, LLC                      7
Daimler Trucks North America, LLC        7
Mack Trucks, Inc.                        7
NAVISTAR, INC                            6
Altec Industries Inc.                    6
Hyundai Motor America                    6
Porsche Cars North America, Inc.         6
Triple E Recreational Vehicles           5
Jayco, Inc.                              5
Thor Motor Coach                         5
DAIMLER TRUCKS NORTH AMERICA             4
General Motors LLC                       4
Navistar, Inc.                           4
Corp. Micro Bird Inc.                    4
Hea

In [14]:
import re

def clean_mfr(name):
    """Normalise a manufacturer name so spelling variants match."""
    s = name.upper()                                                  # ignore case
    s = re.sub(r"[.,]", "", s)                                        # drop commas and dots
    s = re.sub(r"\b(LLC|INC|CORP|CO|COMPANY|LTD)\b", "", s)            # drop legal suffixes
    s = re.sub(r"\s+", " ", s).replace(" )", ")").strip()              # tidy spaces
    return s

bf["mfr"] = bf["MFGNAME"].apply(clean_mfr)
print("Distinct names:", bf["MFGNAME"].nunique(), "->", bf["mfr"].nunique())

# Show every cleaned name that merged more than one original spelling
merged = bf.groupby("mfr")["MFGNAME"].unique()
merged[merged.str.len() > 1]

Distinct names: 141 -> 119


mfr
ALTEC INDUSTRIES                                             [ALTEC INDUSTRIES INC., Altec Industries Inc., Altec Industries, Inc.]
BLUE BIRD BODY                                                                     [BLUE BIRD BODY COMPANY, Blue Bird Body Company]
BMW OF NORTH AMERICA                                                         [BMW OF NORTH AMERICA, LLC, BMW of North America, LLC]
CHRYSLER (FCA US)                                                                   [Chrysler (FCA US LLC), Chrysler (FCA US, LLC)]
DAIMLER TRUCKS NORTH AMERICA    [DAIMLER TRUCKS NORTH AMERICA, Daimler Trucks North America LLC, Daimler Trucks North America, LLC]
ELGIN SWEEPER                                                             [Elgin Sweeper Company Inc., Elgin Sweeper Company, Inc.]
FOREST RIVER                                                                               [FOREST RIVER, INC., Forest River, Inc.]
GENERAL MOTORS                                                      [GEN

In [15]:
# 1) Main analysis set: battery + fire recalls (small, safe to commit)
keep = ["mfr", "MFGNAME", "RCDATE", "year", "POTAFF", "battery_type",
        "hv_comp", "lv_comp", "battery_text", "DESC_DEFECT", "CONEQUENCE_DEFECT"]
bf[keep].to_csv("../data/processed/battery_fire_recalls.csv")

# 2) All vehicle recalls with flags (needed for the classifier in Part 2)
rec.to_csv("../data/processed/vehicle_recalls.csv")

print("Saved:", len(bf), "battery-fire recalls and", len(rec), "vehicle recalls")

Saved: 347 battery-fire recalls and 13721 vehicle recalls


In [16]:
bf["mfr"].value_counts().head(10)

mfr
FORD MOTOR                         20
BMW OF NORTH AMERICA               19
DAIMLER TRUCKS NORTH AMERICA       18
VOLKSWAGEN GROUP OF AMERICA        15
FOREST RIVER                       14
GENERAL MOTORS                     12
CHRYSLER (FCA US)                  11
NAVISTAR                           10
ALTEC INDUSTRIES                   10
JAGUAR LAND ROVER NORTH AMERICA    10
Name: count, dtype: int64